# 15 — SDK Inference Driver (Milestone M7 exit criterion)

The thin **inference driver** over the packaged pipeline: everything below uses
**only `import semigraph`** — no notebook-local helpers. Training/build-time concerns
(ingestion, graph construction, evaluation) live in the SDK and its CLI
(`semigraph ingest | build-graph | eval`); this notebook drives **inference only**
(hybrid retrieval + grounded generation) against the already-built graph.
Notebooks 00–14 stay untouched as the historical record.

**Exit criterion:** fresh environment → `uv pip install .` → this notebook reproduces
**Query D** (the M5 flagship): *"What dependencies connect Meta's AI infrastructure
plans to Nvidia, TSMC, HBM suppliers, and export controls?"* with citations.

Prerequisites: Neo4j Desktop running with the M5 graph loaded; `.env` in the repo root.
Cost: one Sonnet answering call (cents).

In [ ]:
import os
from pathlib import Path

# run from the repo root so .env and data/ resolve (the SDK itself is path-agnostic)
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import semigraph
from semigraph.config import get_settings
from semigraph.embeddings import Embedder
from semigraph.graph.client import get_driver
from semigraph.retrieval.answerer import answer

settings = get_settings()
driver = get_driver(settings)
embedder = Embedder()
print(f"semigraph {semigraph.__version__} | Neo4j @ {settings.neo4j_uri} | {embedder.name}")

## Query D — multi-hop supply-chain + export-control reasoning

In [ ]:
QUERY_D = (
    "What dependencies connect Meta's AI infrastructure plans to Nvidia, TSMC, "
    "HBM suppliers, and export controls?"
)

result = answer(QUERY_D, driver, embedder, strategy="hybrid")

print(result["answer"])
print("\nCitations:")
for c in result["citations"]:
    print(f"  - {c}")

## M7 exit assertion — answer is grounded, cited, and multi-hop

In [ ]:
# --- M7 EXIT assertion cell ---
answer_text = result["answer"].lower()

assert len(result["answer"]) > 200, "suspiciously short answer"
assert result["citations"], "Query D answer must carry citations"

# every cited chunk id must exist as an EvidenceSpan in the graph (0-hallucination bar from M6)
with driver.session() as s:
    n_valid = s.run(
        "MATCH (e:EvidenceSpan) WHERE e.chunk_id IN $ids RETURN count(e) AS n",
        ids=list(result["citations"]),
    ).single()["n"]
assert n_valid == len(result["citations"]), "hallucinated citation id(s)"

# the multi-hop chain: the answer must connect the ecosystem, not just mention Meta
for required in ("nvidia", "tsmc"):
    assert required in answer_text, f"expected '{required}' in the Query D answer"
assert ("hbm" in answer_text or "memory" in answer_text), "expected HBM/memory suppliers in the answer"
assert ("export" in answer_text), "expected export-control linkage in the answer"

driver.close()
print(f"M7 COMPLETE — Query D reproduced via the semigraph SDK: "
      f"{len(result['citations'])} valid citations, all graph-grounded")